# Store Sales — Time Series Forecasting
## Giải pháp 4-way Minimum-Variance Ensemble

**Bộ cạnh tranh:** [Kaggle Store Sales – Time Series Forecasting](https://www.kaggle.com/competitions/store-sales-time-series-forecasting)  
**Mục tiêu:** Dự báo doanh số 16 ngày cho 1782 chuỗi (54 cửa hàng × 33 nhóm hàng) tại Ecuador.  
**Metric:** RMSLE (Root Mean Squared Log Error) — thấp hơn = tốt hơn.

---

## Kiến trúc tổng thể

```
Dữ liệu thô (train.csv, test.csv, oil.csv, holidays_events.csv, stores.csv, transactions.csv)
        │
        ├── [LEG 1] Darts Family (6 biến thể cây GBDT)  ──────────── σ ≈ 0.382
        │     ├── base       : LightGBM depth-6, lags 56/7/365/730
        │     ├── deeper     : LightGBM depth-8
        │     ├── xgb        : XGBoost + LightGBM
        │     ├── subsampled : 3-seed bagging
        │     ├── weighted   : sample_weight theo căn bậc hai
        │     └── cat_deep   : CatBoost depth-8
        │
        ├── [LEG 2] LGBM-v8 (Direct Multi-step)  ─────────────────── σ ≈ 0.477
        │     └── 16 mô hình riêng (h=1..16), features phong phú + oil/holiday FE
        │
        ├── [LEG 3] Chronos-2 Covariates (Foundation LLM)  ────────── σ ≈ 0.398
        │     └── Amazon Chronos-2, conditioned on onpromotion + oil + holiday
        │
        └── [LEG 4] TSMixer Tuned (Neural MLP-Mixer)  ─────────────── σ ≈ 0.382
              └── Global darts TSMixer, 1782 chuỗi, HID=128, BLK=8, 30 epochs
                        │
                        ▼
        4-way Minimum-Variance Ensemble (log1p space)
        weights ≈ [0.400, 0.011, 0.207, 0.382] (fam/v8/cov/tsm)
                        │
                        ▼
        submission_fam_cov_v8_tsmTuned_4way.csv  →  RMSLE ≈ 0.37485
```

---
## Chương 1 — Cài đặt & Dữ liệu

### 1.1 Môi trường và đường dẫn

In [1]:
import os, sys, time, subprocess, shlex
from pathlib import Path
import pandas as pd
import numpy as np

# --- Locate repo root ---
def _find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "config.yaml").exists() and (p / "src").exists():
            return p
    raise RuntimeError("Không tìm thấy thư mục gốc repo!")

REPO = _find_root(Path.cwd())
SRC  = REPO / "src"
DATA = REPO / "data"
SUBMISSIONS = REPO / "submissions"
SUBMISSIONS.mkdir(exist_ok=True)

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from store_sales.config import get_config
from store_sales.ensemble import blend

cfg = get_config()
print(f"REPO        = {REPO}")
print(f"SRC         = {SRC}")
print(f"DATA        = {DATA}")
print(f"SUBMISSIONS = {SUBMISSIONS}")

REPO        = /Users/ngohoangkhactuong/Kaggle/store-sales-forecasting
SRC         = /Users/ngohoangkhactuong/Kaggle/store-sales-forecasting/src
DATA        = /Users/ngohoangkhactuong/Kaggle/store-sales-forecasting/data
SUBMISSIONS = /Users/ngohoangkhactuong/Kaggle/store-sales-forecasting/submissions


### 1.2 Kiểm tra dữ liệu đầu vào

In [2]:
expected = ["train.csv", "test.csv", "stores.csv", "oil.csv",
            "holidays_events.csv", "transactions.csv"]
present  = [f for f in expected if (DATA / f).exists()]
missing  = [f for f in expected if f not in present]
print(f"Tìm thấy: {present}")
if missing:
    raise FileNotFoundError(f"Thiếu file: {missing} — đặt vào thư mục data/")
else:
    print("Tất cả file dữ liệu đã sẵn sàng.")

Tìm thấy: ['train.csv', 'test.csv', 'stores.csv', 'oil.csv', 'holidays_events.csv', 'transactions.csv']
Tất cả file dữ liệu đã sẵn sàng.


### 1.3 Công tắc chạy

| Biến | Giá trị | Ý nghĩa |
|---|---|---|
| `RUN_LEGS` | `True` | Huấn luyện lại mô hình nếu file CSV chưa tồn tại |
| `RUN_LEGS` | `False` | Chỉ chạy bước Blend từ các file CSV đã có |
| `FORCE_RETRAIN` | `True` | Bắt buộc train lại dù file CSV đã tồn tại |

> **Khuyến nghị:** `RUN_LEGS=True, FORCE_RETRAIN=False` để chỉ train những mô hình chưa có file.

In [3]:
RUN_LEGS     = False   # True = train legs nếu chưa có CSV; False = blend-only
FORCE_RETRAIN = False  # True = train lại dù CSV đã tồn tại
TIMINGS = {}           # dict ghi thời gian train từng mô hình

def run(cmd: str, produces: str = "", *, force: bool = FORCE_RETRAIN) -> None:
    """Chạy một lệnh shell. Bỏ qua nếu file đầu ra đã tồn tại và force=False."""
    label = produces or cmd.split()[0]
    out_path = SUBMISSIONS / produces if produces else None

    if out_path and out_path.exists() and not force:
        print(f"  [exists] {produces}  (bỏ qua)")
        return

    if not RUN_LEGS:
        print(f"  [skip]   {label}  (RUN_LEGS=False)")
        return

    args = shlex.split(cmd)
    # Trên máy này không có lệnh `python` trên PATH — chỉ có `python3`.
    # Thay token `python` đứng đầu bằng đúng interpreter đang chạy notebook.
    if args and args[0] == "python":
        args[0] = sys.executable

    env = {**os.environ, "PYTHONPATH": str(SRC) + os.pathsep + os.environ.get("PYTHONPATH", "")}
    print(f"  [run]    {label}")
    t0 = time.time()
    subprocess.run(args, env=env, cwd=str(REPO), check=True)
    elapsed = time.time() - t0
    TIMINGS[label] = elapsed
    print(f"  [done]   {label}  ({elapsed/60:.1f} min)")

print(f"RUN_LEGS={RUN_LEGS}, FORCE_RETRAIN={FORCE_RETRAIN}")

RUN_LEGS=False, FORCE_RETRAIN=False


### 1.4 Cài thư viện (nếu cần)

In [4]:
if RUN_LEGS:
    print("Kiểm tra thư viện...")
    try:
        import lightgbm, xgboost, catboost, darts
        print("lightgbm, xgboost, catboost, darts — đã cài sẵn")
    except ImportError as e:
        print(f"Thiếu thư viện: {e}. Đang cài...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "lightgbm", "xgboost", "catboost", "darts"], check=True)
        print("Cài xong")
else:
    print("RUN_LEGS=False — bỏ qua kiểm tra thư viện.")

RUN_LEGS=False — bỏ qua kiểm tra thư viện.


---
## Chương 2 — Huấn luyện các mô hình

> Mỗi cell dưới đây train một nhánh (leg). Cell chỉ thực sự chạy khi:
> - `RUN_LEGS = True`, **VÀ**
> - File CSV đầu ra chưa tồn tại (hoặc `FORCE_RETRAIN = True`)

### 2.1 LEG 1 — Darts Family (6 biến thể cây GBDT)

Cùng một khung Darts nhưng khác nhau về thuật toán, độ sâu cây, và chiến lược lấy mẫu.  
Mỗi biến thể huấn luyện **4 mô hình lồng nhau** (lags 56/7/365/730 ngày) rồi lấy trung bình.

| Biến thể | Thuật toán | Đặc điểm | File output |
|---|---|---|---|
| `base` | LightGBM | Cấu hình mặc định + FE (oil+holiday) | `submission_darts_lgbm.csv` |
| `deeper` | LightGBM | depth=8 | `submission_darts_lgbm_deeper.csv` |
| `xgb` | XGBoost+LGBM | Bổ sung XGBoost 4 configs | `submission_darts_xgb_lgbm.csv` |
| `subsampled` | LightGBM | Trung bình 3 seeds | `submission_darts_lgbm_subsampled_3seed.csv` |
| `weighted` | LightGBM | Đánh trọng số √ theo thời gian | `submission_darts_lgbm_w.csv` |
| `cat_deep` | CatBoost | depth=8 | `submission_darts_cat_deeper.csv` |

In [5]:
run("python -m store_sales.cli train darts-family --variant base",
    produces="submission_darts_lgbm.csv")

  [exists] submission_darts_lgbm.csv  (bỏ qua)


In [6]:
run("python -m store_sales.cli train darts-family --variant deeper",
    produces="submission_darts_lgbm_deeper.csv")

  [exists] submission_darts_lgbm_deeper.csv  (bỏ qua)


In [7]:
run("python -m store_sales.cli train darts-family --variant xgb",
    produces="submission_darts_xgb_lgbm.csv")

  [exists] submission_darts_xgb_lgbm.csv  (bỏ qua)


In [8]:
run("python -m store_sales.cli train darts-family --variant subsampled",
    produces="submission_darts_lgbm_subsampled_3seed.csv")

  [exists] submission_darts_lgbm_subsampled_3seed.csv  (bỏ qua)


In [9]:
run("python -m store_sales.cli train darts-family --variant weighted",
    produces="submission_darts_lgbm_w.csv")

  [exists] submission_darts_lgbm_w.csv  (bỏ qua)


In [10]:
run("python -m store_sales.cli train darts-family --variant cat_deep",
    produces="submission_darts_cat_deeper.csv")

  [exists] submission_darts_cat_deeper.csv  (bỏ qua)


### 2.2 LEG 2 — LGBM-v8 (Direct Multi-step Forecasting)

Thay vì dự đoán toàn bộ 16 ngày cùng một lúc, v8 huấn luyện **16 mô hình LightGBM độc lập**
(một mô hình cho mỗi ngày trong horizon h=1..16).  
Điều này giúp mỗi mô hình học được pattern riêng của từng ngày dự báo.

**Features đặc biệt của v8:**
- Lag ngắn (1-21 ngày) + lag dài (28, 35, ..., 364 ngày)  
- Rolling windows (7, 14, 28, 56, 84, 168 ngày)  
- EWM half-life (7, 28 ngày)  
- Oil dynamics (biến động giá dầu)  
- Holiday distance (khoảng cách đến ngày lễ gần nhất)  
- Promotional features (tương tác khuyến mãi × ngày trong tuần)

In [11]:
run("python -m store_sales.cli train lgbm-v8 --suffix reg",
    produces="submission_v8_reg.csv")

  [exists] submission_v8_reg.csv  (bỏ qua)


### 2.3 LEG 3 — Chronos-2 với Covariates (Foundation LLM)

Mô hình ngôn ngữ lớn dành cho chuỗi thời gian của Amazon, được điều kiện hoá trên:
- **`onpromotion`**: biến động khuyến mãi (known-future covariate)  
- **`oil`**: giá dầu WTI (past-only covariate)  
- **`holiday`**: cờ ngày lễ quốc gia (known-future covariate)

> **Yêu cầu đặc biệt:** Chronos-2 cần **virtual environment riêng** với Python 3.11  
> và `chronos-forecasting==2.2.2` vì có conflict dependency với các thư viện khác.
>
> **Thời gian chạy:** ~15-30 phút (CPU inference)  
> **Lần đầu:** Tự động tải model ~2GB từ HuggingFace về cache.

**Thiết lập venv:**
```bash
python3.11 -m venv .venv_chronos2
.venv_chronos2/bin/pip install "chronos-forecasting==2.2.2" numpy pandas pyarrow
```

In [12]:
# Kiểm tra .venv_chronos2 đã tồn tại chưa
venv_python = REPO / ".venv_chronos2/bin/python"
if not venv_python.exists():
    print("   Chưa có .venv_chronos2!")
    print("   Chạy lệnh sau để tạo:")
    print("   python3.11 -m venv .venv_chronos2")
    print("   .venv_chronos2/bin/pip install 'chronos-forecasting==2.2.2' numpy pandas pyarrow")
else:
    import subprocess as _sp
    r = _sp.run([str(venv_python), "-c", "from chronos import Chronos2Pipeline; print('OK')"],
                capture_output=True, text=True, cwd=str(REPO))
    if r.returncode == 0:
        print(".venv_chronos2 sẵn sàng — Chronos2Pipeline import OK")
    else:
        print("Lỗi import Chronos2:", r.stderr[:200])

.venv_chronos2 sẵn sàng — Chronos2Pipeline import OK


In [13]:
# Train Chronos-2 với oil + holiday covariates
# Tạo ra: submissions/submission_chronos2_cov_promo_oil_hol.csv
run(f"{venv_python} -m store_sales.cli train chronos2-cov --oil --holiday --suffix _promo_oil_hol",
    produces="submission_chronos2_cov_promo_oil_hol.csv")

  [exists] submission_chronos2_cov_promo_oil_hol.csv  (bỏ qua)


### 2.4 LEG 4 — TSMixer (Global Neural Forecaster)

Mô hình mạng nơ-ron MLP-Mixer huấn luyện **toàn cục** trên tất cả 1782 chuỗi cùng một lúc.  
Covariates: khuyến mãi, giá dầu, lịch, ngày lễ.

**Cấu hình tuned:** `HID=128, FF=128, BLK=8, DROP=0.2, LR=1e-3, epochs=30`  
**Scheduler:** CosineAnnealingLR

In [14]:
# TSMixer: dùng file đã train sẵn nếu tồn tại (tiết kiệm 6+ tiếng)
run("python -m store_sales.cli train tsmixer --epochs 30 --cpu --mps",
    produces="submission_tsmixer_tuned.csv")

  [exists] submission_tsmixer_tuned.csv  (bỏ qua)


---
## Chương 3 — Ensemble & Kết quả

### 3.1 Kiểm tra đầy đủ các file submission

Trước khi trộn, xác nhận tất cả file CSV của các nhánh đã tồn tại trong `submissions/`.

In [15]:
get_config.cache_clear()
cfg = get_config()

needed = (
    list(cfg.ensemble.family_files.values()) +
    list(cfg.ensemble.leg_files.values())
)

all_ok = True
for fname in needed:
    path = SUBMISSIONS / fname
    exists = path.exists()
    size_kb = f"{path.stat().st_size//1024} KB" if exists else "---"
    status = "success" if exists else "fail"
    print(f"  {status}  {fname:<52} {size_kb}")
    if not exists:
        all_ok = False

if all_ok:
    print("\nTất cả file sẵn sàng — có thể chạy bước blend!")
else:
    print("\nCòn thiếu file — hãy train lại các mô hình bị thiếu ở Chương 2.")

  success  submission_darts_lgbm.csv                            708 KB
  success  submission_darts_lgbm_deeper.csv                     708 KB
  success  submission_darts_xgb_lgbm.csv                        708 KB
  success  submission_darts_lgbm_subsampled_3seed.csv           708 KB
  success  submission_darts_cat_deeper.csv                      709 KB
  success  submission_darts_lgbm_w.csv                          708 KB
  success  submission_chronos2_cov_promo_oil_hol.csv            497 KB
  success  submission_v8_reg.csv                                722 KB
  success  submission_tsmixer_tuned.csv                         716 KB

Tất cả file sẵn sàng — có thể chạy bước blend!


### 3.2 Trọng số blend & RMSLE ước lượng

Hàm `build_fourway()` tính **minimum-variance weights** dựa trên ma trận hiệp phương sai  
được ước lượng từ các điểm số σ_i (RMSLE trên Kaggle Leaderboard) và mức độ khác biệt  
giữa các dự đoán của từng nhánh.

**Công thức:** `Cov_ij = (σ_i² + σ_j² − D_ij²) / 2`  
trong đó `D_ij = RMS(log_pred_i − log_pred_j)` — hai nhánh khác biệt nhiều → bổ sung tốt hơn.

In [16]:
get_config.cache_clear()
cfg = get_config()
ens = cfg.ensemble

# Bảng sigma từng nhánh
print("=== Sigma (RMSLE standalone trên Kaggle LB) ===\n")
rows = []
for m in ens.family_files:
    rows.append({"leg": f"family.{m}", "sigma": ens.family_sigma[m],
                 "file": ens.family_files[m]})
for leg in ens.leg_files:
    rows.append({"leg": leg, "sigma": ens.leg_sigma[leg],
                 "file": ens.leg_files[leg]})
df_sigma = pd.DataFrame(rows).sort_values("sigma")
display(df_sigma.style.format({"sigma": "{:.5f}"}))

# Tính trọng số blend 4-way
print("\n=== 4-way Blend Weights & Estimated RMSLE ===\n")
r = blend.build_fourway()
table = pd.DataFrame([{
    "leg": leg,
    "sigma": ens.leg_sigma.get(leg, ens.family_sigma.get(leg, "-")),
    "weight": f"{w:.4f}",
} for leg, w in zip(r["legs"], r["weights"])])
display(table)
print(f"\nEstimated math_LB = {r['blend_rmsle_formula']:.5f}")
print("(Điểm thực tế trên Kaggle thường chênh ±0.0002 so với ước lượng này)")

=== Sigma (RMSLE standalone trên Kaggle LB) ===



,leg,sigma,file
8,tsmixer,0.38191,submission_tsmixer_tuned.csv
2,family.xgb,0.38202,submission_darts_xgb_lgbm.csv
1,family.deeper,0.38269,submission_darts_lgbm_deeper.csv
3,family.sub3,0.38298,submission_darts_lgbm_subsampled_3seed.csv
0,family.base,0.38337,submission_darts_lgbm.csv
5,family.weighted,0.38604,submission_darts_lgbm_w.csv
4,family.cat_deep,0.38659,submission_darts_cat_deeper.csv
6,chronos2-cov,0.39772,submission_chronos2_cov_promo_oil_hol.csv
7,lgbm-reg,0.47682,submission_v8_reg.csv



=== 4-way Blend Weights & Estimated RMSLE ===



,leg,sigma,weight
0,family,0.38120,0.4142
1,chronos2-cov,0.39772,0.2051
2,lgbm-reg,0.47682,0.0080
3,tsmixer,0.38191,0.3726



Estimated math_LB = 0.37478
(Điểm thực tế trên Kaggle thường chênh ±0.0002 so với ước lượng này)


### 3.3 Tạo file nộp cuối cùng

Cell này chạy `store_sales.cli blend build` để:
1. Tính minimum-variance weights
2. Trộn 4 nhánh trong log1p-space
3. Ghi ra file nộp cuối cùng

In [17]:
env = {**os.environ, "PYTHONPATH": str(SRC) + os.pathsep + os.environ.get("PYTHONPATH", "")}

print("=== Bước 1: Build final submission ===")
build_result = subprocess.run(
    [sys.executable, "-m", "store_sales.cli", "blend", "build"],
    env=env, cwd=str(REPO), capture_output=False, text=True
)

print("\n=== Bước 2: Verify byte-exact reproduction ===")
verify_result = subprocess.run(
    [sys.executable, "-m", "store_sales.cli", "blend", "verify"],
    env=env, cwd=str(REPO), capture_output=False, text=True
)

# Kiểm tra file đầu ra
get_config.cache_clear()
cfg = get_config()
out_file = SUBMISSIONS / cfg.ensemble.out_file
if out_file.exists():
    sub = pd.read_csv(out_file)
    print(f"\nFile nộp đã tạo: {out_file.name}")
    print(f"   Rows: {len(sub):,} | Mean sales: {sub['sales'].mean():.2f} | Max: {sub['sales'].max():.0f}")
else:
    print("Không tìm thấy file nộp!")

=== Bước 1: Build final submission ===
weights family=0.4142 chronos2-cov=0.2051 lgbm-reg=0.0080 tsmixer=0.3726
math_LB = 0.37478  [reference actual LB 0.37418]
diff family<->tsmixer = 0.126 (family-level redundancy watch)
Wrote submission_fam_cov_v8_tsmTuned_4way.csv  (rows=28512, max sales=14339)
Wrote submission_family.csv  (rows=28512)

=== Bước 2: Verify byte-exact reproduction ===
weights family=0.4142 chronos2-cov=0.2051 lgbm-reg=0.0080 tsmixer=0.3726
math_LB = 0.37478  [reference actual LB 0.37418]
diff family<->tsmixer = 0.126 (family-level redundancy watch)
VERIFY max|delta| = 1.819e-12  ->  EXACT MATCH

File nộp đã tạo: submission_fam_cov_v8_tsmTuned_4way.csv
   Rows: 28,512 | Mean sales: 433.92 | Max: 14339


### 3.4 Thời gian huấn luyện từng mô hình

In [18]:
if TIMINGS:
    items = sorted(TIMINGS.items(), key=lambda kv: kv[1], reverse=True)
    df_time = pd.DataFrame([
        {"model": k, "minutes": round(v/60, 1), "seconds": round(v, 0)}
        for k, v in items
    ])
    display(df_time)
    print(f"\nTổng thời gian train: {sum(TIMINGS.values())/60:.1f} phút")
else:
    print("Không có mô hình nào được train trong phiên này (tất cả đã dùng file sẵn có).")

Không có mô hình nào được train trong phiên này (tất cả đã dùng file sẵn có).


---
## Chương 4 — Hướng dẫn nộp bài

### 4.1 File nộp cuối cùng

| File | Mô tả | RMSLE ước tính |
|---|---|---|
| `submission_fam_cov_v8_tsmTuned_4way.csv` | **4-way Ensemble** (nộp bài chính) | **≤ 0.37485** |
| `submission_family.csv` | Family sub-blend (6 GBDT) | ≈ 0.38183 |
| `submission_darts_lgbm.csv` | LightGBM base | ≈ 0.38337 |
| `submission_tsmixer_tuned.csv` | TSMixer | ≈ 0.38191 |
| `submission_chronos2_cov_promo_oil_hol.csv` | Chronos-2 oil+hol | ≈ 0.39772 |

### 4.2 Nộp file lên Kaggle

```bash
# Cài Kaggle CLI (nếu chưa có)
pip install kaggle

# Nộp file ensemble
kaggle competitions submit \
    -c store-sales-time-series-forecasting \
    -f submissions/submission_fam_cov_v8_tsmTuned_4way.csv \
    -m "4-way minvar ensemble: family+chronos2-oilhol+v8+tsmixer"
```

### 4.3 Cập nhật sigma sau khi nộp

Sau khi nộp từng file lên Kaggle và nhận điểm RMSLE thực tế,  
cập nhật vào `config.yaml` phần `ensemble.family_sigma` / `ensemble.leg_sigma`,  
rồi chạy lại **Chương 3** để tính lại weights tối ưu.

### 4.4 Kết luận

| Tiêu chí | Kết quả |
|---|---|
| **Metric đạt được** | RMSLE ≤ 0.37485 (public LB) |
| **Phương pháp cốt lõi** | 4-way minimum-variance ensemble |
| **Mô hình quan trọng nhất** | Darts Family (weight ≈ 40%) + TSMixer (≈ 38%) |
| **Nguồn cải thiện chính** | Chronos-2 với Oil + Holiday covariates |

> **Insight:** Các mô hình "yếu" như Chronos-2 (σ=0.40) và LGBM-v8 (σ=0.48) lại có đóng góp lớn  
> vì **lỗi của chúng không tương quan** với các mô hình cây — giúp blend triệt tiêu được phần sai số chung.